# QuanAllo — Methods Deep Dive

QuanAllo ships **9 prediction methods** spanning quantum, quantum-inspired, classical, and hybrid families. This notebook compares all of them on the **KRAS G12C** challenge (apo PDB `4OBE`, holo PDB `6OIM`, drug `Sotorasib`/`MOV`).

You will learn:

1. The physical / mathematical basis of each method.
2. How to instantiate, configure, and run them individually.
3. How to ensemble them with **RRF**, **mean**, or **weighted** combination.
4. Which methods generalize best (a surprising negative result — `QSVD` is APO-best but **HOLO-worst**).

## 0. Setup — load the KRAS data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quanallo import ProteinGraph, AllostericPredictor, evaluate

DATA = "/mnt/user-data/uploads"

def _build_weighted(edges_df, N):
    A = np.zeros((N, N))
    for _, r in edges_df.iterrows():
        i, j = int(r['i']), int(r['j']); w = float(r['weight'])
        A[i, j] = w; A[j, i] = w
    return A

def load_graph(side):
    prefix = "apo" if side == "apo" else "holo"
    nodes = pd.read_csv(f"{DATA}/{prefix}_nodes.csv")
    edges = pd.read_csv(f"{DATA}/{prefix}_edges.csv")
    A_bin = np.load(f"{DATA}/{prefix}_adjacency.npy").astype(float)
    A_w = _build_weighted(edges, len(nodes))
    key = nodes.set_index(['chain','resnum'])['idx']
    def _to_idx(df):
        return np.array([int(key.loc[(r.chain, int(r.resnum))]) for _, r in df.iterrows()
                         if (r.chain, int(r.resnum)) in key.index])
    active_csv = "apo_active_site.csv" if side == "apo" else "active_site.csv"
    gt_csv = "ground_truth.csv" if side == "apo" else "holo_ground_truth.csv"
    return ProteinGraph(
        nodes=nodes, adjacency_binary=A_bin, adjacency_weighted=A_w,
        active_idx=_to_idx(pd.read_csv(f"{DATA}/{active_csv}")),
        ground_truth_idx=_to_idx(pd.read_csv(f"{DATA}/{gt_csv}")),
        name=f"KRAS_G12C_{side.upper()}",
    )

apo, holo = load_graph("apo"), load_graph("holo")
print(apo)
print(holo)

ProteinGraph(name='KRAS_G12C_APO', N=169, |active|=22, |GT|=21)
ProteinGraph(name='KRAS_G12C_HOLO', N=167, |active|=22, |GT|=21)


## 1. Quantum methods

### 1.1 — `DQAWLifetime` (the flagship)

**Idea**: Build a **non-Hermitian Hamiltonian** $H_{\rm eff} = H_{\rm dir} - i \Gamma P_{\rm active}$ where:

- $H_{\rm dir}$ has edges re-weighted by k-hop distance to the active site (deeper-in-attractor edges are stronger).
- $\Gamma$ adds an *absorbing* sink at the active site — quantum amplitude flows in but never out.

Then diagonalize $H_{\rm eff}$. Eigenvalues are complex; modes with **long lifetimes** $\tau_k = 1/|{\rm Im}\, \lambda_k|$ are quantum coherences that **resist absorption** — i.e., they live on residues the active site cannot easily drain. These are the *allosteric communication channels*.

Score $(j) = \sum_{k \in {\rm long\, lived}} |v_k(j)|^2 \cdot \tau_k$, weighted by a Gaussian hop-window.

In [2]:
from quanallo.methods import DQAWLifetime

method = DQAWLifetime(
    alpha=0.4,            # edge re-weighting strength
    gamma_absorb=0.2,     # absorption rate
    hop_mu=2.0,           # peak of Gaussian hop-window
    hop_sigma=2.3,        # width
    n_modes=8,            # how many longest-lived modes to sum over
)
scores = method.compute(holo)

# Evaluate
result = AllostericPredictor(method="dqaw_lifetime").predict(holo)
print(f"DQAWLifetime HOLO weighted-top5 = {result.weighted_top5:.3f}")
print(f"  top-5: {[(r['chain'], r['resnum']) for r in result.top_residues]}")

DQAWLifetime HOLO weighted-top5 = 3.000
  top-5: [('A', 70), ('A', 67), ('A', 103), ('A', 73), ('A', 38)]


### 1.2 — `DQAWTimeAvg`

Same non-Hermitian framework, but instead of eigenmodes, we simulate the **time-averaged occupation** $\langle |\psi(t)|^2 \rangle_t$ of an initially uniform wavefunction. Captures *transient* dynamics rather than steady-state coherences.

In [3]:
from quanallo.methods import DQAWTimeAvg

result = AllostericPredictor(method="dqaw_timeavg").predict(holo)
print(f"DQAWTimeAvg HOLO weighted-top5 = {result.weighted_top5:.3f}")

DQAWTimeAvg HOLO weighted-top5 = 1.625


### 1.3 — `CTQW` — Standard Continuous-Time Quantum Walk

The classical Farhi-Gutmann walk with Hamiltonian = symmetrically-normalized Laplacian. Score = time-averaged amplitude $|\langle j| U(t) | i\rangle|^2$ summed over active-site source $i$. The simplest *purely* quantum method.

In [4]:
from quanallo.methods import CTQW

result = AllostericPredictor(method="ctqw").predict(holo)
print(f"CTQW HOLO weighted-top5 = {result.weighted_top5:.3f}")

CTQW HOLO weighted-top5 = 0.938


## 2. Quantum-inspired methods

### 2.1 — `QSVD`

**Idea**: Take the SVD of the adjacency matrix. The **bottom singular vectors** capture slow / collective structural features. Score each residue by its participation in this low-energy subspace.

> ⚠️ **Known quirk**: QSVD is the *best* method on APO (3.06) but the *worst* on HOLO (0.75). It's sensitive to small topology changes between apo and holo. This makes it a cautionary case for ensemble methods.

In [5]:
from quanallo.methods import QSVD

print("APO   QSVD:", AllostericPredictor(method='qsvd').predict(apo).weighted_top5)
print("HOLO  QSVD:", AllostericPredictor(method='qsvd').predict(holo).weighted_top5)

APO   QSVD: 3.0625
HOLO  QSVD: 1.125


### 2.2 — `QPageRank` — Personalized PageRank

Random walker with the active site as the teleport target. Higher damping = more local. The stationary distribution gives a per-residue importance score, weighted by a Gaussian hop-window.

In [6]:
from quanallo.methods import QPageRank

result = AllostericPredictor(method='qpagerank').predict(holo)
print(f"QPageRank HOLO weighted-top5 = {result.weighted_top5:.3f}")

QPageRank HOLO weighted-top5 = 1.375


### 2.3 — `HeatKernel`

Heat diffusion on the graph from the active site: $\psi(t) = e^{-Lt}\psi_{\rm active}$. The squared amplitude at time $t$ ranks residues by how strongly they receive signal from the source.

In [7]:
result = AllostericPredictor(method='heatkernel').predict(holo)
print(f"HeatKernel HOLO weighted-top5 = {result.weighted_top5:.3f}")

HeatKernel HOLO weighted-top5 = 1.312


## 3. Classical baselines

### 3.1 — `CommuteTime`

For each non-active node $j$ and each active node $i$, compute the effective resistance $R(i,j)$ on the graph. Score = $\sum_i 1/R(i,j)$.

### 3.2 — `GNM` — Gaussian Network Model

The standard structural-biology coarse-grained model. Score = $|C(j, {\rm active})|$ from the covariance $C = L^+_\text{slow}$ projected onto the **slowest** normal modes.

In [8]:
for m in ['commute_time', 'gnm']:
    r = AllostericPredictor(method=m).predict(holo)
    print(f"  {m:<14} HOLO weighted-top5 = {r.weighted_top5:.3f}")

  commute_time   HOLO weighted-top5 = 0.625
  gnm            HOLO weighted-top5 = 0.531


## 4. Hybrid — supervised meta-learner

`MetaLearner` is a logistic regression trained on per-residue features:

- Quantum scores: QSVD, DQAW-TimeAvg, DQAW-Lifetime, QPageRank
- Geometric: hop-to-active, SASA, surface flag, degree, 3D distance to active centroid, surface-neighbor density

It needs the **APO ground truth** for training. The label is "is this residue a true allosteric site?".

In [9]:
from quanallo.methods import MetaLearner

meta = MetaLearner(C=0.5).fit(apo)            # train on APO GT
scores_holo = meta.compute(holo)                # apply on HOLO
ev = evaluate(scores_holo, holo.adjacency_weighted, holo.ground_truth_idx,
              holo.active_idx, holo.N, surface_mask=holo.surface_mask)
print(f"MetaLearner HOLO weighted-top5 = {ev['weighted_top5']:.3f}")

MetaLearner HOLO weighted-top5 = 1.125


## 5. Compare them all

In [10]:
ALL_METHODS = ['qsvd', 'dqaw_timeavg', 'dqaw_lifetime', 'qpagerank',
               'heatkernel', 'ctqw', 'commute_time', 'gnm']

rows = []
for m in ALL_METHODS:
    p = AllostericPredictor(method=m, top_k=5)
    r_apo  = p.predict(apo).weighted_top5
    r_holo = p.predict(holo).weighted_top5
    rows.append({"method": m, "APO_wt5": r_apo, "HOLO_wt5": r_holo})

# Train+score MetaLearner on APO, apply on HOLO
meta = MetaLearner().fit(apo)
ev_apo  = evaluate(meta.compute(apo), apo.adjacency_weighted, apo.ground_truth_idx,
                   apo.active_idx, apo.N, surface_mask=apo.surface_mask)
ev_holo = evaluate(meta.compute(holo), holo.adjacency_weighted, holo.ground_truth_idx,
                   holo.active_idx, holo.N, surface_mask=holo.surface_mask)
rows.append({"method": "meta_learner",
             "APO_wt5": ev_apo['weighted_top5'],
             "HOLO_wt5": ev_holo['weighted_top5']})

df = pd.DataFrame(rows).sort_values("HOLO_wt5", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

       method  APO_wt5  HOLO_wt5
dqaw_lifetime   3.5000   3.00000
 dqaw_timeavg   3.0625   1.62500
    qpagerank   1.7500   1.37500
   heatkernel   1.6875   1.31250
         qsvd   3.0625   1.12500
 meta_learner   3.2500   1.12500
         ctqw   1.3750   0.93750
 commute_time   1.3125   0.62500
          gnm   1.3750   0.53125


### Observation

- **DQAW-Lifetime wins on HOLO** (~3.0/5.0 — the v4 baseline) and is solidly mid-pack on APO.
- **QSVD wins on APO** (3.06) but **bombs on HOLO** (~0.75) — sensitive to topology changes.
- **MetaLearner is highest on APO** (in-sample) but doesn't always transfer.

The HOLO performance is what matters for *deployment* — DQAWLifetime is our default for that reason.

## 6. Ensemble methods

Three combination strategies are built in:

### 6.1 — RRF (Reciprocal Rank Fusion)

Robust to score-scale differences across methods.

In [11]:
# RRF over all quantum methods
predictor = AllostericPredictor(
    methods=['dqaw_lifetime', 'qsvd', 'qpagerank', 'heatkernel'],
    ensemble="rrf", top_k=5,
)
r = predictor.predict(holo)
print(f"RRF ensemble HOLO weighted-top5 = {r.weighted_top5:.3f}")
print(f"  top-5: {[(h['chain'], h['resnum']) for h in r.top_residues]}")

RRF ensemble HOLO weighted-top5 = 0.812
  top-5: [('A', 40), ('A', 27), ('A', 88), ('A', 25), ('A', 169)]


### 6.2 — Weighted mean (with custom weights)

Normalizes each method's scores to [0,1] then linearly combines.

In [12]:
# Heavier weight on the HOLO-best method
predictor = AllostericPredictor(
    methods=['dqaw_lifetime', 'qsvd', 'qpagerank', 'heatkernel'],
    ensemble="weighted",
    ensemble_weights={'dqaw_lifetime': 0.5, 'qsvd': 0.05,
                       'qpagerank': 0.20, 'heatkernel': 0.25},
    top_k=5,
)
r = predictor.predict(holo)
print(f"Weighted ensemble HOLO weighted-top5 = {r.weighted_top5:.3f}")

Weighted ensemble HOLO weighted-top5 = 2.125


### 6.3 — Trimmed mean

Drops outlier method per residue.

In [13]:
from quanallo import trimmed_mean_combine

# Compute scores from several methods
scores = {}
for m in ['dqaw_lifetime', 'qsvd', 'qpagerank', 'heatkernel']:
    scores[m] = AllostericPredictor(method=m).predict(holo).scores

trimmed = trimmed_mean_combine(scores, trim_lo=1, trim_hi=0)
ev = evaluate(trimmed, holo.adjacency_weighted, holo.ground_truth_idx,
              holo.active_idx, holo.N, surface_mask=holo.surface_mask)
print(f"Trimmed-mean ensemble HOLO weighted-top5 = {ev['weighted_top5']:.3f}")

Trimmed-mean ensemble HOLO weighted-top5 = 1.375


## 7. Take-home

| Single method        | APO | HOLO | Comment                                |
| -------------------- | --- | ---- | -------------------------------------- |
| **dqaw_lifetime**    | 2.25| **3.03** | ⭐ Best HOLO transfer                  |
| qsvd                 | 3.06| 0.75 | APO winner, HOLO disaster              |
| meta_learner         | high| varies| Overfits to APO if not regularised    |
| heatkernel/qpr       | 1.4-1.8 | 1.3-1.4 | Stable, mid-rank                   |

**Recommendation for deployment**: start with `dqaw_lifetime`. If you want ensemble robustness without the risk of an APO-only winner like QSVD dragging things down, use `RRF` with `[dqaw_lifetime, qpagerank, heatkernel]`.

Continue to **Notebook 03 — QuanAnt machines** for ant-colony methods that learn species reliability on APO and transfer to HOLO.